# Lesson 20 Lab — Benchmarking Latency, Throughput, and Workloads

**Puzzle:** Which benchmark result survives a change in prompt length, output length, or concurrency?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A single tokens-per-second value collapses the workload. A useful benchmark retains request shapes, warm-up state, raw samples, successful token counts, and the distinction between offline throughput and online latency.


## 0. Predict before running

1. Predict which batch has the highest output tokens/s.
2. Check whether all requests generated the same token limit.
3. State why this is not a concurrency benchmark.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The native experiment warms one engine and runs batch sizes 1, 4, and 8 across fixed prompts. It retains every wall-time sample and reports useful output throughput plus request throughput.

- Workload identity includes both input and output lengths.
- Raw samples make percentile and noise checks recomputable.
- Throughput and tail latency can move in opposite directions.


## 2. Derive the mechanism

Offline throughput measures completed work over a closed batch; online serving adds arrivals, queueing, TTFT, ITL, cancellation, and network. Larger batches can raise GPU utilization while increasing a request's wait. Prompt and output tokens must remain separate because they exercise Prefill and Decode differently.

### Mechanism at a glance

```mermaid
flowchart LR
  W["prompt/output distribution"] --> B["benchmark cell"]
  C["batch or arrival pattern"] --> B
  E["engine + model + GPU"] --> B
  B --> R["raw per-request/per-run records"]
  R --> T["TTFT + ITL + latency + throughput"]
  T --> G["SLO attainment, not one headline number"]
```

### Walk it step by step

1. **Freeze the workload.** Record input/output tokens and arrival or batch policy.
2. **Warm intentionally.** Separate cold compilation/startup from steady state.
3. **Keep raw records.** Retain samples and successful token counts.
4. **Report the right scope.** Do not turn an offline batch rate into an online SLO.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 20
LESSON_TITLE = 'Benchmarking Latency, Throughput, and Workloads'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260832
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | batch size one |
| Candidate | batch sizes four and eight |
| Held constant | model, prompt template, maximum output, greedy sampling, warm-up, repeats, and GPU |
| Measurements | raw elapsed samples, prompt/output tokens, request/s, output tokens/s, and memory |
| Evidence | `native-backend` |

**Experiment:** Warm one native engine and measure repeated closed batches at three batch sizes.


## 5. Inspect the experiment code

The engine stays alive across the sweep. Each batch uses distinct prompts of similar length, and the artifact stores samples rather than only a rounded mean.

Do not execute until the code matches the frozen table.


In [2]:
llm=LLM(**base_engine_args(max_model_len=1024,max_num_seqs=16)); params=SamplingParams(temperature=0.0,max_tokens=24,seed=SEED)
llm.generate(["Warm."],SamplingParams(temperature=0.0,max_tokens=2),use_tqdm=False); rows={}; torch.cuda.reset_peak_memory_stats()
for batch in (1,4,8):
    samples=[]; prompt_tokens=output_tokens=0
    for repeat in range(3):
        prompts=[f"Request {repeat}-{i}: explain one vLLM metric." for i in range(batch)]
        tick=time.perf_counter(); outputs=llm.generate(prompts,params,use_tqdm=False); samples.append(time.perf_counter()-tick)
        prompt_tokens=sum(len(x.prompt_token_ids or []) for x in outputs); output_tokens=sum(len(x.outputs[0].token_ids) for x in outputs)
    median=statistics.median(samples); rows[str(batch)]={"samples_s":samples,"median_s":median,
        "p95_s":percentile(samples,.95),"prompt_tokens":prompt_tokens,"output_tokens":output_tokens,
        "requests_s":batch/median,"output_tokens_s":output_tokens/median}
metrics={"batches":rows,"repeats":3,"peak_allocated_mib":torch.cuda.max_memory_allocated()/2**20}
analysis=(f"Batch 1/4/8 measured {rows['1']['output_tokens_s']:.1f}/{rows['4']['output_tokens_s']:.1f}/"
          f"{rows['8']['output_tokens_s']:.1f} output tokens/s in a warmed closed workload. Raw samples "
          "are retained; online TTFT/ITL are outside scope.")


INFO 08-13 00:22:56 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260832, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:22:56 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:22:56 [model.py:1883] Using max model len 1024


INFO 08-13 00:22:56 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:22:56 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=652915) 2026-08-13 00:23:08,655 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json
(EngineCore pid=652915) 2026-08-13 00:23:08,655 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=652915) 2026-08-13 00:23:08,715 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=652915) 2026-08-13 00:23:08,721 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=652915) INFO 08-13 00:23:09 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=652915) INFO 08-13 00:23:09 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.71 s


(EngineCore pid=652915) WARNING 08-13 00:23:10 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=652915) WARNING 08-13 00:23:10 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=652915) INFO 08-13 00:23:10 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=652915) INFO 08-13 00:23:10 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=652915) INFO 08-13 00:23:10 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:23:10 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Batch-1 output tok/s | 126.2/s |
| Batch-4 output tok/s | 467.4/s |
| Batch-8 output tok/s | 918.3/s |
| Batch-1 requests/s | 5.3/s |
| Batch-8 requests/s | 38.3/s |
| Peak allocated | 0.000 MiB |


## 7. Explain the result

Batch 1/4/8 measured 126.2/467.4/918.3 output tokens/s in a warmed closed workload. Raw samples are retained; online TTFT/ITL are outside scope.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 20, "title": 'Benchmarking Latency, Throughput, and Workloads', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The native sweep establishes offline batching behavior for three frozen cells; it cannot be generalized to online traffic without new evidence.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 20,
  "title": "Benchmarking Latency, Throughput, and Workloads",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260832
  },
  "evidence_label": "native-backend",
  "metrics": {
    "batches": {
      "1": {
        "samples_s": [
          0.19944166205823421,
          0.19021221715956926,
          0.1879363670013845
        ],
        "median_s": 0.19021221715956926,
        "p95_s": 0.19851871756836773,
        "prompt_tokens": 13,
        "output_tokens": 24,
        "requests_s": 5.2572858617230604,
        "output_tokens_s": 126.17486068135345
      },
      "4": {
        "samples_s": [
          0.22377016488462687,
          0.20532395131886005,
          0.20539495069533587
        ],
        "median_s": 0.20539495069533587,
        "p95_s": 0.2219326434656977

## 9. Make the bounded decision

> The native sweep establishes offline batching behavior for three frozen cells; it cannot be generalized to online traffic without new evidence.

**Acceptance/rollback:** Use a benchmark row only for the workload cell it names; production promotion additionally requires open-loop service replay and tail gates.

**Failure analysis:** Closed-loop batches create no queue and may benefit from cache/compilation warm-up. Small sample counts and generated-length variation can bias rates.


## 10. Extend the evidence

Use `vllm bench serve` or a timestamped client to sweep request rate, concurrency, prompt/output distributions, streaming, and SLO attainment.

The full boundary and references are in [`README.md`](README.md).
